In [1]:
# Processing data and configurations
import xarray as xr
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.ops import cascaded_union
import os

from datetime import datetime
import glob

In [2]:
def preprocess_rean(xdf):
    """Adjust longitude bounds and round coordinates."""
    xdf['longitude'] = np.round(xdf.longitude, 3)
    xdf['latitude'] = np.round(xdf.latitude, 3)
    
    if xdf.longitude.values.min() >= 0:
        xdf = roll_xdf(xdf)
    return xdf

def roll_xdf(xdf):
    """Roll longitude from 0-360 to -180-180."""
    xdf = xdf.assign_coords(longitude=(((xdf.longitude + 180) % 360) - 180))
    return xdf

def get_geometries(cut_on, shapefile=None):
    """Retrieve geometries for given region or bounding box coordinates."""
    if cut_on == 'europe':
        geometries = [-25, 32, 48.5, 75]
    elif cut_on == 'SADC':
        geometries = [2, -37, 60, 7]
    elif isinstance(cut_on, list):
        geometries = cut_on
    else:   
        if not shapefile:
            geometries = get_boundaries(cut_on.capitalize(), continent=False)
        else:
            geometries = gpd.read_file(shapefile)
            geometries = geometries[geometries['NAME_EN'] == cut_on]['geometry']
    return geometries

def get_boundaries(name, continent=False):
    """Retrieve boundary geometries by name."""
    world_path = '../world_map/'
    world = gpd.read_file(world_path)
    
    if continent:
        world = world.query('REGION_UN == @name')
        boundary = gpd.GeoSeries(cascaded_union(world['geometry']))
    elif name in world['NAME_EN'].values:
        region_mask = world[world['NAME_EN'] == name]
        polygons = region_mask['geometry']
        boundary = gpd.GeoSeries(cascaded_union(polygons)) if (polygons.geom_type == 'MultiPolygon').any() else gpd.GeoSeries(polygons)
    return boundary

In [3]:
def load_data(data_path, component):
    xdf = xr.open_mfdataset(data_path + f'10m_{component}_*.nc', engine='netcdf4',preprocess=preprocess_rean)
    return xdf


def include_time_coord(ds):
    """Ensure time coordinate inclusion."""
    if 'time' not in ds.coords:
        ds = ds.assign_coords(time=('time', pd.date_range(datetime.now(), periods=ds.dims['time'])))
    return ds

def normalize_naming(xdf):
    """Standardize variable names as required."""
    # Placeholder normalization, as naming conventions may vary
    return xdf.rename({'var_name_placeholder': 'variable_standard_name'}) 

def aggregate_daymean(xdf, var_name):
    """Compute daily mean for given variable."""
    xdf = xdf.rename({'valid_time':'time'})
    xdf_daymean = xdf[var_name].resample(time='1D').mean()
    return xdf_daymean

def cut_xdf(xdf, geometries):
    
    xdf = xdf.sortby('latitude')
    """Subset xdf using provided geometries."""
    # Assuming geometries are bounding box [lon_min, lat_min, lon_max, lat_max]
    # Assuming 'geometries' is a DataFrame with bounding box columns
    lon_min = geometries.bounds['minx'].values[0]  # Extract minimum longitude
    lat_min = geometries.bounds['miny'].values[0]  # Extract minimum latitude
    lon_max = geometries.bounds['maxx'].values[0]  # Extract maximum longitude
    lat_max = geometries.bounds['maxy'].values[0]  # Extract maximum latitude
    
    xdf_cut = xdf.sel(longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max))
    return xdf_cut

In [4]:
country = 'cuba'## FILL WITH THE COUNTRY NAME e.g. cuba
component = 'v' ## FILL WITH THE COMPONENT OF THE WIND e.g. u,v

# Processing data and configurations
shapefile_path = '../world_map/ne_50m_admin_0_countries.shp'
geometries = get_geometries(f'{country.capitalize()}', shapefile_path)

ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/preprocess_conda/share/proj failed


In [5]:
## 
data_path = '../data/cds_downloads/' ## FILL WITH THE PATH of ERA5LAND data
output_path = f'../data/processed_data/{country}/' ## FILL WITH THE PATH of desidered OUTPUT PATH
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)


In [6]:
def main(variable, data_sources, start_date, end_date, loading_area, aggregation_mode, overwrite=False, data_path=data_path, output_path=output_path):
    # Step 1: Define date range and load data
    xdf = load_data(data_path, component)
    
    # Step 2: Apply geographic masking if specified
    if loading_area:
        geometries = get_geometries(loading_area)
        xdf = cut_xdf(xdf, geometries)
        print(f"Data masked to the specified loading area: {loading_area}")

    # Step 3: Aggregate data by daily mean if required
    if aggregation_mode == 'daymean':
        xdf = aggregate_daymean(xdf, var_name=variable)
        print("Data aggregated by daily mean.")

    # Step 4: Save output if overwrite is enabled
    output_filename = f"{output_path}{variable}_{start_date}_{end_date}_processed.nc"
    if overwrite or not os.path.exists(output_filename):
        os.makedirs(output_path, exist_ok=True)
        print(xdf)
        xdf = xdf.drop('number')
        xdf.to_netcdf(output_filename)
        print(f"Data written to {output_filename}")
    else:
        print(f"File already exists and overwrite is set to False: {output_filename}")

In [7]:
# Parameters (Example Usage)
main(
    variable=f'{component}10',
    data_sources='era5land',
    start_date='1992-01',
    end_date='1992-01',
    loading_area=country,
    aggregation_mode='daymean',
    data_path=data_path,
    output_path=output_path,
    overwrite=True
)

/tmp/ipykernel_624929/1959858302.py:42: ShapelyDeprecationWarning: The 'cascaded_union()' function is deprecated. Use 'unary_union()' instead.
  boundary = gpd.GeoSeries(cascaded_union(polygons)) if (polygons.geom_type == 'MultiPolygon').any() else gpd.GeoSeries(polygons)
/tmp/ipykernel_624929/3946059427.py:21: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  xdf = xdf.drop('number')


Data masked to the specified loading area: cuba
Data aggregated by daily mean.
<xarray.DataArray 'v10' (time: 31, latitude: 32, longitude: 79)> Size: 313kB
dask.array<stack, shape=(31, 32, 79), dtype=float32, chunksize=(1, 32, 75), chunktype=numpy.ndarray>
Coordinates:
    number     int64 8B ...
  * latitude   (latitude) float64 256B 19.9 20.0 20.1 20.2 ... 22.8 22.9 23.0
  * longitude  (longitude) float64 632B -84.8 -84.7 -84.6 ... -77.2 -77.1 -77.0
  * time       (time) datetime64[ns] 248B 1992-01-01 1992-01-02 ... 1992-01-31
Attributes: (12/32)
    GRIB_paramId:                             166
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      24311
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_units:                         